# Feature eng

In [0]:
# carregando tabela de vendas
df_sales = spark.table("adventure_works.marts.fct_sales_for_ds_analysis")

# exibe amostra
df_sales.show(10)


In [0]:
# carregando tabela de endereço
df_addres = spark.table("adventure_works.marts.dim_address")\
    .select("address_id",
            "country_name",
            "city",
            "country_region_name")

# exibe amostra
df_addres.show(3)

In [0]:
df_merge = df_sales.join(df_addres, on="address_id", how="full")
df_merge.show(15)

In [0]:
# carregando tabela de endereço
df_dates = spark.table("adventure_works.marts.dim_dates")\
    .select("date_day",
            "day_of_month",
            "day_of_week",
            "day_of_week_name",
            "day_of_year",
            "quarter_of_year",
            "year_number",
            "month_of_year"
            )

# exibe amostra
df_dates.show(3)

In [0]:
df_merge = df_merge.join(df_dates, df_merge["date_order_date"] == df_dates["date_day"], how="full")\
             .drop(df_dates["date_day"])
df_merge.show(15)

import pandas as pd
import numpy as np

# df já agregado por loja/região/produto
df = df_merge.sort_values(["store_name","country_region_name","product_subcategory_name","date_order_date"])

# Criando lags
for lag in [1,7,30]:
    df[f"lag_{lag}"] = df.groupby(["store_name","country_region_name","product_subcategory_name"])["total_qty"].shift(lag)

# Criando médias móveis
df["rolling_mean_7"] = df.groupby(["store_name","country_region_name","product_subcategory_name"])["total_qty"].shift(1).rolling(7).mean()
df["rolling_mean_30"] = df.groupby(["store_name","country_region_name","product_subcategory_name"])["total_qty"].shift(1).rolling(30).mean()
df["rolling_std_30"] = df.groupby(["store_name","country_region_name","product_subcategory_name"])["total_qty"].shift(1).rolling(30).std()

# Features de calendário
df["day_of_week"] = df["date_order_date"].dt.dayofweek
df["month"] = df["date_order_date"].dt.month
df["quarter"] = df["date_order_date"].dt.quarter

# Target
df = df.dropna()  # remover linhas iniciais sem lags
X = df.drop(columns=["total_qty","date_order_date"])
y = df["total_qty"]


In [0]:
%pip install xgboost

In [0]:
# Converter colunas categóricas para pandas category
categorical_cols = [
    "address_id", "sales_sk", "product_subcategory_name", 
    "person_type", "person_type_name", "store_name", 
    "country_name", "city", "country_region_name", "day_of_week_name"
]

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Definir a janela de partição por loja/região/produto, ordenada pela data
w = Window.partitionBy(
    "store_name", "country_region_name", "product_subcategory_name"
).orderBy("date_order_date")

# Criar lags (1, 7, 30)
df = df_merge
for lag in [1, 7, 30]:
    df = df.withColumn(f"lag_{lag}", F.lag("product_qty_day", lag).over(w))

# Criar médias móveis e desvio padrão
# Observação: Spark não tem shift(1). Para simular, usamos rowsBetween deslocado (-window, -1).
rolling_7 = w.rowsBetween(-7, -1)
rolling_30 = w.rowsBetween(-30, -1)

df = df.withColumn("rolling_mean_7", F.avg("product_qty_day").over(rolling_7))
df = df.withColumn("rolling_mean_30", F.avg("product_qty_day").over(rolling_30))
df = df.withColumn("rolling_std_30", F.stddev("product_qty_day").over(rolling_30))

# Features de calendário
df = df.withColumn("day_of_week", F.dayofweek("date_order_date") - 1)  # Spark: 1=Domingo
df = df.withColumn("month", F.month("date_order_date"))
df = df.withColumn("quarter", F.quarter("date_order_date"))

# Remover linhas com null (lags iniciais)
df = df.na.drop(subset=["lag_1", "lag_7", "lag_30", "rolling_mean_7", "rolling_mean_30", "rolling_std_30"])

# Definir X e y
X = df.drop("product_qty_day", "date_order_date")
y = df.select("product_qty_day")

# Converter Spark DataFrames para Pandas
X = X.toPandas()
y = y.toPandas()

In [0]:
print(X.shape[0])

In [0]:
import numpy as np
print(y.isna().sum())       # Contar valores nulos
print(np.isinf(y).sum())    # Contar valores infinitos
print(y.describe())         # Conferir valores muito grandes


In [0]:
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
import mlflow
import mlflow.xgboost

# Definir o número de splits
tscv = TimeSeriesSplit(n_splits=5)

mae_scores = []

for fold, (train_index, test_index) in enumerate(tscv.split(X)):
    print(f"Fold {fold + 1}")
    # Separar treino e teste
    X_train, X_test = X.iloc[train_index].copy(), X.iloc[test_index].copy()
    y_train, y_test = y.iloc[train_index].copy(), y.iloc[test_index].copy()

    # Converter colunas categóricas para category (mantendo compatibilidade com XGBoost)
    for col in categorical_cols:
        X_train[col] = X_train[col].astype("category")
        X_test[col]  = X_test[col].astype("category")
    
    # Iniciar run MLflow
    with mlflow.start_run(run_name=f"xgb_fold_{fold+1}") as run:
            
        # Instanciar modelo
        model = XGBRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            tree_method="hist",
            enable_categorical=True
        )

        # Treinar
        print("Treinando...")
        model.fit(X_train, y_train)

        # Previsão
        print("Prevendo...")
        y_pred = model.predict(X_test)

        # Avaliar
        print("Avaliando...")
        mae = mean_absolute_error(y_test, y_pred)
        mae_scores.append(mae)

        # Logar métrica e parâmetros
        mlflow.log_metric("mae", mae)
        mlflow.log_param("n_estimators", 500)
        mlflow.log_param("learning_rate", 0.05)
        mlflow.log_param("max_depth", 6)

        print(f"Fold {fold+1}:")
        print(f"  Train: {train_index[0]} -> {train_index[-1]}")
        print(f"  Test : {test_index[0]} -> {test_index[-1]}")
        print(f"  MAE  : {mae:.4f}")
        print("-"*40)

        # Logar modelo
        model.save_model("xgb_model.json")
        mlflow.xgboost.log_model(model, artifact_path="model")
        mlflow.xgboost.log_model(model, "xgb_model", input_example=X_train.iloc[:3])

print("MAE médio:", sum(mae_scores)/len(mae_scores))

criar uma função para esse modelo
testar segregar o dataframe por produto, região, etc... e verificar o comportamento das métricas do modelo

In [0]:
df_sales_for_store = df_merge.filter(df_merge["person_type"] == "SC")
df_sales_for_store.show(3)

In [0]:
df_sales_for_individual = df_merge.filter(df_merge["person_type"] == "IN")
df_sales_for_individual.show(3)

In [0]:
df_merge.select("product_subcategory_name").distinct().show(1000)

In [0]:
columns = [
            "date_order_date",
            "product_subcategory_name",
            "product_qty_day",
            "person_type",
            "person_type_name",
            "store_name",
            "city",
            "country_name",
            "country_region_name"]
